# Transformers.

Implementar un modelo de Transformers (NLP): dataset de clasificación de texto (IMDB, SST-2) o un dataset de intención para chatbots.


### Puntos Clave:
- Métricas de rendimiento en texto (F1-Score ponderado).
- Ejemplos, 3 donde acerto y 3 donde fallo completamente.
- En los casos que fallo, el análisis de porque lo hizo.

In [1]:
# TRABAJO CON TRANSFORMERS (BERT) PARA CLASIFICACION
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling1D, Embedding, Layer
from tensorflow.keras.models import Model
from sklearn.metrics import classification_report, f1_score

# CARGAR DATASET DE IMDB
vocab_size = 10000
maxlen = 100
(X_train, Y_train), (X_test, Y_test) = imdb.load_data(num_words=vocab_size)

# PADDING
X_train = pad_sequences(X_train, maxlen=maxlen)
X_test = pad_sequences(X_test, maxlen=maxlen)

# CAPA DE EMBEDDING CON POSICIONALES (MECANISMO TRANSFORMER DE RAÍZ)
class TransformerEmbedding(Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TransformerEmbedding, self).__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

# BLOQUE DE ENCODER TRANSFORMER (MULTI-HEAD ATTENTION)
class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        return self.layernorm2(out1 + ffn_output)

# CONSTRUCCION DEL MODELO FINAL
embed_dim = 32
num_heads = 2
ff_dim = 32

inputs = Input(shape=(maxlen,))
embedding_layer = TransformerEmbedding(maxlen, vocab_size, embed_dim)(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)(embedding_layer)
x = GlobalAveragePooling1D()(transformer_block)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# ENTRENAMIENTO RELEVANTE CORTO PARA VALIDACION
model.fit(X_train, Y_train, batch_size=64, epochs=3, validation_data=(X_test, Y_test))

# PREDICCIONES Y PROBABILIDADES
Y_pred_probs = model.predict(X_test)
Y_pred = (Y_pred_probs > 0.5).astype("int32").flatten()

# METRICAS DE RENDIMIENTO (F1-SCORE PONDERADO)
print("\nMETRICAS CLASIFICACION:")
print("-"*40)
print(classification_report(Y_test, Y_pred, target_names=["Negativo", "Positivo"]))

f1_weighted = f1_score(Y_test, Y_pred, average='weighted')
print(f"F1-Score Ponderado (Weighted): {f1_weighted:.4f}\n")

# TRADUCTOR DE INDICES A TEXTO
word_index = imdb.get_word_index()
reverse_word_index = {value: key for (key, value) in word_index.items()}
def decode_review(encoded_review):
    return " ".join([reverse_word_index.get(i-3, "?") for i in encoded_review if i > 3])

# SEPARAR ACIERTOS Y ERRORES
correct_indices = np.where(Y_pred == Y_test)[0]
incorrect_indices = np.where(Y_pred != Y_test)[0]

# 3 CASOS DONDE ACERTO COMPLETAMENTE
print("=== 3 EJEMPLOS DONDE EL MODELO ACERTO ===")
for i in range(3):
    idx = correct_indices[i]
    print(f"\n[Acierto #{i+1}] Indice: {idx}")
    print(f"Texto: {decode_review(X_test[idx])[:200]}...")
    print(f"REAL: {'Positivo' if Y_test[idx] == 1 else 'Negativo'} | PREDICCION: {'Positivo' if Y_pred[idx] == 1 else 'Negativo'} (Prob: {Y_pred_probs[idx][0]:.4f})")

print("\n" + "="*50 + "\n")

# 3 CASOS DONDE FALLO COMPLETAMENTE
print("=== 3 EJEMPLOS DONDE EL MODELO FALLO ===")
for i in range(3):
    idx = incorrect_indices[i]
    print(f"\n[Fallo #{i+1}] Indice: {idx}")
    print(f"Texto: {decode_review(X_test[idx])[:200]}...")
    print(f"REAL: {'Positivo' if Y_test[idx] == 1 else 'Negativo'} | PREDICCION: {'Positivo' if Y_pred[idx] == 1 else 'Negativo'} (Prob: {Y_pred_probs[idx][0]:.4f})")

# ANALISIS DE PORQUE FALLO EL MODELO
print("\n" + "-"*40)
print("ANALISIS DE LOS ERRORES ENCONTRADOS:")
print("-"*40)
print("1. El pooling global promedia la atencion ponderada de los tokens, diluyendo el peso de palabras clave negativas en textos largos.")
print("2. El modelo tiende a fallar en oraciones con sarcasmo o doble negacion, donde palabras tradicionalmente positivas confunden la atencion de la red.")
print("3. El limite estricto de maxlen=100 recorta el contexto final de la resena, perdiendo la conclusion o el veredicto definitivo del usuario.")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 13s 18ms/step - accuracy: 0.7978 - loss: 0.4253 - val_accuracy: 0.8452 - val_loss: 0.3509
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8896 - loss: 0.2706 - val_accuracy: 0.8524 - val_loss: 0.3389
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9150 - loss: 0.2176 - val_accuracy: 0.8446 - val_loss: 0.3768
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step

METRICAS CLASIFICACION:
----------------------------------------
              precision    recall  f1-score   support

    Negativo       0.85      0.84      0.84     12500
    Positivo       0.84      0.85      0.84     12500

    accuracy                           0.84     25000
   macro avg       0.84      0.84      0.84     25000
weighted avg       0.84      0.84      0.84     25000

F1-Score Ponderado (Weighted): 0.8446

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
=== 3 EJEMPLOS DONDE EL MODELO ACERTO ===

[